In [1]:
import pandas as pd
import os
import sys

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss


c:\Users\mohme\Desktop\projet_recettes\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
from src.embedding import generate_embeddings_and_save_faiss
# Importe la fonction `generate_embeddings_and_save_faiss` que tu viens de définir dans `src/embedding.py`.

print("Modules importés avec succès.")

Modules importés avec succès.


In [5]:
input_combined_recipes_path = "../data/processed/all_recipes_combined_processed3.csv"
output_embeddings_dir = "../data/embeddings" # C'est le répertoire pour FAISS et les métadonnées

print(f"Chemin du fichier d'entrée : {input_combined_recipes_path}")
print(f"Répertoire de sortie des embeddings FAISS : {output_embeddings_dir}")

Chemin du fichier d'entrée : ../data/processed/all_recipes_combined_processed3.csv
Répertoire de sortie des embeddings FAISS : ../data/embeddings


In [6]:
try:
    df_combined_recipes = pd.read_csv(input_combined_recipes_path)
    # Tente de charger le fichier CSV spécifié dans un DataFrame Pandas.
    print(f"Dataset chargé avec succès. Nombre total de recettes : {len(df_combined_recipes)}")
    print("Premières lignes du dataset chargé :")
    display(df_combined_recipes.head())
    # Affiche les 5 premières lignes du DataFrame pour vérifier le chargement.
    print("\nInformations sur le dataset :")
    df_combined_recipes.info()
    # Affiche un résumé des informations du DataFrame (colonnes, types de données, non-null counts).

except FileNotFoundError:
    # Si le fichier spécifié par input_combined_recipes_path n'est pas trouvé.
    print(f"Erreur : Le fichier {input_combined_recipes_path} n'a pas été trouvé.")
    print("Assurez-vous d'avoir exécuté le notebook 02_recipe_preprocessing.ipynb et que le chemin est correct.")
    sys.exit(1) # Quitte l'exécution si le fichier n'est pas trouvé
    # Affiche un message d'erreur et arrête le script car le fichier d'entrée est manquant.



Dataset chargé avec succès. Nombre total de recettes : 232111
Premières lignes du dataset chargé :


,name,id,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,ingredients_text,steps_text,document
0,"""It'S Maaaaa-Gic!"" Moroccan Carrots",NaN,NaN,"['marocaine', 'north african']",NaN,NaN,['Cook the carrots in a small amount of salted...,NaN,"['1 pound skinny carrots, scraped and cut into...",NaN,"1 pound skinny carrots, scraped and cut into l...",1. Cook the carrots in a small amount of salte...,"\n Title: ""It'S Maaaaa-Gic!"" Moroccan Carro..."
1,"""Moroccan Guacamole"" Toasts With Fried Egg",NaN,NaN,"['marocaine', 'north african']",NaN,NaN,"['Make ""guacamole:"" Squeeze or scoop avocado f...",NaN,"['1 ripe avocado, halved and pitted', '1/8 to ...",NaN,"1 ripe avocado, halved and pitted\n1/8 to 1/4 ...","1. Make ""guacamole:"" Squeeze or scoop avocado ...","\n Title: ""Moroccan Guacamole"" Toasts With ..."
2,0 carb 0 cal gummy worms,283618.0,45.0,"['60-minutes-or-less', 'time-to-make', 'course...","[384.7, 0.0, 0.0, 70.0, 159.0, 0.0, 6.0]",15.0,"['stir all ingredients until dissolved', 'pour...",these are delicious and guilt free!\r\n\r\n*if...,"['sugar-free jello', 'gelatin', 'boiling water']",3.0,sugar-free jello\ngelatin\nboiling water,1. stir all ingredients until dissolved\n2. po...,\n Title: 0 carb 0 cal gummy worms\n T...
3,0 fat chunky watermelon salsa,176996.0,45.0,"['60-minutes-or-less', 'time-to-make', 'course...","[57.7, 0.0, 39.0, 0.0, 2.0, 0.0, 4.0]",6.0,['using a small sharp knife cut peel and white...,this is fantastic served with grilled seafood ...,"['lime', 'watermelon', 'cucumber', 'green onio...",8.0,lime\nwatermelon\ncucumber\ngreen onion\ncilan...,1. using a small sharp knife cut peel and whit...,\n Title: 0 fat chunky watermelon salsa\n ...
4,0 point ice cream only 1 ingredient,480558.0,125.0,"['time-to-make', 'course', 'main-ingredient', ...","[304.1, 13.0, 121.0, 3.0, 13.0, 9.0, 19.0]",5.0,"['1) peel your bananas first', '2) cut them in...",yummy!! (got this recipe from \r\nhttp://www.t...,"['bananas', 'peanut butter', 'honey']",3.0,bananas\npeanut butter\nhoney,1. 1) peel your bananas first\n2. 2) cut them ...,\n Title: 0 point ice cream only 1 ingredi...



Informations sur le dataset :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232111 entries, 0 to 232110
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   name              232111 non-null  object 
 1   id                230184 non-null  float64
 2   minutes           230184 non-null  float64
 3   tags              232111 non-null  object 
 4   nutrition         230184 non-null  object 
 5   n_steps           230184 non-null  float64
 6   steps             232111 non-null  object 
 7   description       225258 non-null  object 
 8   ingredients       232111 non-null  object 
 9   n_ingredients     230184 non-null  float64
 10  ingredients_text  232111 non-null  object 
 11  steps_text        232111 non-null  object 
 12  document          232111 non-null  object 
dtypes: float64(4), object(9)
memory usage: 23.0+ MB


In [7]:
print("Démarrage de la génération des embeddings et de la création de l'index FAISS...")
# Affiche un message indiquant le début du processus.
generate_embeddings_and_save_faiss(
    df=df_combined_recipes,
    model_name='all-MiniLM-L6-v2', # Modèle léger et efficace
    text_column='document',
    output_dir=output_embeddings_dir
)
# Appelle la fonction que tu as définie dans src/embedding.py.
# Les arguments sont passés : le DataFrame, le nom du modèle, la colonne de texte, et le répertoire de sortie.
print("Processus de génération des embeddings et de création de l'index FAISS terminé.")
# Affiche un message de fin.

print("\n--- Tes embeddings et l'index FAISS sont maintenant prêts pour la recherche de similarité ! ---")
# Message final pour indiquer la prochaine étape.

Démarrage de la génération des embeddings et de la création de l'index FAISS...
Modèle all-MiniLM-L6-v2 chargé.
Génération des embeddings pour 232111 documents...


Batches: 100%|██████████| 7254/7254 [4:20:37<00:00,  2.16s/it]  


Embeddings générés. Forme : (232111, 384)
Index FAISS sauvegardé dans : ../data/embeddings\recipe_index.faiss
Métadonnées (DataFrame) sauvegardées dans : ../data/embeddings\recipe_metadata.csv
Processus de génération des embeddings et de création de l'index FAISS terminé.

--- Tes embeddings et l'index FAISS sont maintenant prêts pour la recherche de similarité ! ---
